# 16 — Ensemble across the 5 same-preprocessing rung-3+rung-4 variants

**Roadmap item 3** (`project_dat_parkinson_strategic_roadmap.md`): rung 4's
five training-protocol experiments each failed to individually *replace*
the rung-3 CNN -- but that's the wrong frame for whether they're useful as
**ensemble members**. Five mechanistically different training regimes
(family-oversampling, cosine LR, augmentation, class_weight, plus rung-3
itself) produce decorrelated errors, which is exactly the precondition an
ensemble benefits from. All checkpoints are already trained (25 each,
5 seeds x 5 folds x 5 protocols = 125 total) and every OOF array is already
on disk -- this notebook costs zero new GPU time, just `np.mean` over
arrays that exist.

**Denoising (`rung4_denoise_*`) is excluded on purpose**: it used a
different preprocessing pipeline (`config.USE_NLM_DENOISING=True`, a
separate volume cache) and its own gate result was a genuine null (see
`project_dat_parkinson_rung4_gate_review.md` -- mean delta -0.0026, sign
flips twice across repeats). The other four variants are training-loop-only
changes -- byte-identical inference preprocessing to rung 3 -- so they can
be ensembled with rung 3 with no new code.

**Discipline (Opus's explicit caution, given this project's own rung-4
gate-overfitting history)**: there are 2^5=32 possible subsets of
{rung3, familybias, lrsched, augment, classweight}. Searching that space
and reporting whichever subset scores best would repeat the exact mistake
rung 4's gate review caught. **This notebook makes exactly ONE
pre-registered comparison**: the equal-weight average of all 5 variants
(25 repeat-level OOF arrays) vs. notebook 14's rung-3-only 5-way ensemble
(already scored: log loss=0.4127, AUROC=0.8915, ECE=0.0316). A per-variant
breakdown is printed as a diagnostic only -- it must NOT be used to
cherry-pick a subset after the fact.

**Data handling**: loads real row-level labels and OOF prediction arrays,
so per the AI-assistant data rule (`README.md`) this is **[RUN ME]** — run
it yourself, share back only the printed aggregate numbers. CPU-only, no
GPU, no volume cache -- milliseconds.

In [ ]:
# [RUN ME] -- loads real row-level labels + existing OOF prediction arrays
# from all 5 non-denoise variants. CPU-only, no GPU, no volume cache --
# self-contained, does not assume any earlier cell/notebook ran in this
# kernel session.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd

import config
import evaluate

labels_df = pd.read_csv(config.TRAIN_LABELS_PATH)
family_df = pd.read_csv(config.DATA_PROCESSED / "baseline_features.csv")[
    [config.UID_COLUMN, "inplane_family"]
]
labeled_df = labels_df.merge(family_df, on=config.UID_COLUMN, how="inner").reset_index(drop=True)
labels = labeled_df[config.TARGET_COLUMN].tolist()
y_true = np.array(labels)

repeat_seeds = list(range(config.SEED, config.SEED + 5))
VARIANTS = {
    "rung3": "rung3",
    "familybias": "rung4_familybias",
    "lrsched": "rung4_lrsched",
    "augment": "rung4_augment",
    "classweight": "rung4_classweight",
    # rung4_denoise deliberately excluded -- different preprocessing pipeline, see intro cell
}

oof_by_variant = {
    name: [np.load(config.DATA_PROCESSED / f"{prefix}_oof_seed{s}.npy") for s in repeat_seeds]
    for name, prefix in VARIANTS.items()
}
print(f"{len(VARIANTS)} variants x {len(repeat_seeds)} seeds = "
      f"{len(VARIANTS) * len(repeat_seeds)} repeat-level OOF arrays loaded")

In [ ]:
# [RUN ME] (no new data access -- uses the arrays loaded above).
# THE single pre-registered comparison: equal-weight average of all 25
# repeat-level OOF arrays (5 variants x 5 seeds) vs. notebook 14's
# rung-3-only 5-way ensemble. No other comparison in this cell feeds a
# selection decision.
all_oof_arrays = [oof for arrays in oof_by_variant.values() for oof in arrays]
all_variants_ensemble_oof = np.mean(all_oof_arrays, axis=0)

scores = evaluate.combined_score(y_true, all_variants_ensemble_oof)
print(f"all-5-variants ensemble (25 arrays): log loss={scores['log_loss']:.4f}  "
      f"AUROC={scores['auroc']:.4f}  ECE={scores['ece']:.4f}")
print("notebook 14 reference -- rung3-only ensemble (5 arrays): "
      "log loss=0.4127  AUROC=0.8915  ECE=0.0316")
print(f"\ndelta (all-variants - rung3-only): {scores['log_loss'] - 0.4127:+.4f} log loss")
print("THE DECISION RULE: if this delta is negative, adopt all 5 variants as the "
      "production ensemble composition. If positive/flat, keep rung3-only -- do NOT "
      "search subsets of the 5 variants looking for a better combination (see intro cell).")

In [ ]:
# [RUN ME] (no new data access -- uses the arrays loaded above).
# DIAGNOSTIC ONLY -- per-variant-family mean OOF score, informational.
# Do not use this to cherry-pick a subset; the decision was already made
# by the single comparison in the cell above.
print("per-variant 5-seed pooled log loss (diagnostic only, not for selection):")
for name, arrays in oof_by_variant.items():
    variant_ensemble = np.mean(arrays, axis=0)
    ll = evaluate.log_loss_score(y_true, variant_ensemble)
    print(f"  {name:12s} log loss={ll:.4f}")

**What we're looking for:** does the equal-weight ensemble of all 5
same-preprocessing training-protocol variants (25 repeat-level OOF arrays)
beat the rung-3-only 5-way ensemble by more than noise -- confirming that
rung 4's "failed" experiments are still valuable as decorrelated ensemble
members even though none of them individually replaced the CNN?

**What we found:** all-5-variants ensemble (25 arrays): log loss=0.4022,
AUROC=0.8984, ECE=0.0360. Rung3-only ensemble (5 arrays, notebook 14):
log loss=0.4127, AUROC=0.8915, ECE=0.0316. **Delta: -0.0105 log loss**,
moving in the same direction on AUROC too (+0.0069) -- a real, coherent
improvement, not a calibration-only artifact (both discrimination and the
pooled metric move together). ECE ticks up slightly (0.0316 -> 0.0360) but
stays far below the uncalibrated blend's 0.0747 from notebook 14 -- not a
concern. Diagnostic breakdown (not used for selection, but consistent with
the decision): familybias (0.4022) and lrsched (0.4034) individually
already beat rung3's own pooled 0.4127, augment (0.4082) close behind,
classweight (0.4197) weakest but still contributes to the ensemble's error
decorrelation.

**Decision: ADOPT all 5 variants as the production ensemble composition**
(the pre-registered delta is negative, per the decision rule above -- no
subset search was performed).

**Next step:** redo notebook 15's calibration + logit-space blend
re-tuning (roadmap items 2+4) against this new 25-array ensemble's OOF
instead of the rung-3-only 5-array ensemble -- T_cnn/T_baseline/w need to
be re-fit against the composition actually being adopted, not carried over
from the rung3-only fit. That recalibrated, all-variants candidate is what
should go into `submission_src/main.py`/
`src/submission.py::combine_predictions` for the next real submission.
Roadmap item 5 (drop early stopping, train on 100% of the outer-train
fold), if pursued, should target this same 5-variant composition. See
`project_dat_parkinson_strategic_roadmap.md`.